# 01 · Data Cleaning & Preprocessing
**MarketLens: Amazon product performance intelligence (U.S. marketplace)**

The raw file `amazon.csv` is a scrape of **Amazon India** (amazon.in, January 2023): 1,465 rows, 16 columns, all stored as text.
This notebook:
1. Loads the data and checks it (types, missing values, duplicates, bad values)
2. Parses currency, percentage and count strings into numbers (₹, `%`, Indian digit grouping such as `1,07,687`)
3. Fixes corrupt values (a `|` in `rating`, missing `rating_count`)
4. Merges duplicate ASINs (the same product appears under several search pages)
5. Splits the category path into a hierarchy and maps it to **U.S. Amazon category names**
6. Converts INR prices to **U.S. market prices** in USD
7. Builds text-derived features, then drops PII and columns we don't need
8. Saves `data/interim/amazon_us_clean.csv`

> **Note on localisation:** the scrape is from India, but the product targets the U.S. market. We don't just convert currency
> (₹399 → $4.80 isn't a believable U.S. price for a branded cable). We apply a documented **U.S. market price multiplier** and
> snap prices to U.S. `.99` price points. All assumptions are listed in one place below.

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "amazon.csv"
OUT = ROOT / "data" / "interim"
OUT.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(42)   # one seed for every approximation, so runs are reproducible

## 1. Load & first look

In [2]:
raw = pd.read_csv(RAW, dtype=str)
print(raw.shape)
raw.head(3)

(1465, 16)


,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...


In [3]:
info = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "non_null": raw.notna().sum(),
    "nulls": raw.isna().sum(),
    "unique": raw.nunique(),
    "example": raw.iloc[0].str.slice(0, 45),
})
info

,dtype,non_null,nulls,unique,example
product_id,str,1465,0,1351,B07JW9H4J1
product_name,str,1465,0,1337,Wayona Nylon Braided USB to Lightning Fast Ch
category,str,1465,0,211,Computers&Accessories|Accessories&Peripherals
discounted_price,str,1465,0,550,₹399
actual_price,str,1465,0,449,"₹1,099"
discount_percentage,str,1465,0,92,64%
rating,str,1465,0,28,4.2
rating_count,str,1463,2,1143,"24,269"
about_product,str,1465,0,1293,High Compatibility : Compatible With iPhone 1
user_id,str,1465,0,1194,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRB"


In [4]:
print("Fully duplicated rows :", raw.duplicated().sum())
print("Duplicated product_id :", raw.product_id.duplicated().sum())
print("Unique products       :", raw.product_id.nunique())
print("Odd rating values     :", sorted(v for v in raw.rating.unique() if not re.fullmatch(r"\d(\.\d)?", str(v))))

Fully duplicated rows : 0
Duplicated product_id : 114
Unique products       : 1351
Odd rating values     : ['|']


**Findings**
- Every column is text: prices carry `₹` and thousands separators, discounts carry `%`.
- `rating` holds one corrupt value: `|`.
- `rating_count` has 2 nulls.
- No row is an exact duplicate, but **114 `product_id`s repeat**. The same ASIN was scraped from several search pages, with slightly different review snapshots.
- `user_id`, `user_name` and `review_id` are comma-joined lists of ~8 reviewers each. They're PII and not useful for financial modelling.
- `img_link` and `product_link` are URLs we keep only for the front-end.

## 2. Parse numeric columns

In [5]:
def to_number(s: pd.Series) -> pd.Series:
    """'₹1,07,687' / '64%' / '24,269' -> float. Strips any non-numeric char except '.'"""
    return pd.to_numeric(s.str.replace(r"[^0-9.]", "", regex=True).replace("", np.nan), errors="coerce")

df = raw.copy()
df["price_inr"]      = to_number(df["discounted_price"])
df["list_price_inr"] = to_number(df["actual_price"])
df["discount_pct"]   = to_number(df["discount_percentage"])
df["rating"]         = pd.to_numeric(df["rating"], errors="coerce")
df["rating_count"]   = to_number(df["rating_count"])

df[["price_inr", "list_price_inr", "discount_pct", "rating", "rating_count"]].describe().T

,count,mean,std,min,25%,50%,75%,max
price_inr,1465.0,3125.310874,6944.304394,39.0,325.0,799.0,1999.0,77990.0
list_price_inr,1465.0,5444.990635,10874.826864,39.0,800.0,1650.0,4295.0,139900.0
discount_pct,1465.0,47.691468,21.635905,0.0,32.0,50.0,63.0,94.0
rating,1464.0,4.096585,0.291674,2.0,4.0,4.1,4.3,5.0
rating_count,1463.0,18295.541353,42753.864952,2.0,1186.0,5179.0,17336.5,426973.0


### 2.1 Consistency checks

In [6]:
# The discount should equal 1 - price/list (rounded). Check how far off it is.
implied = (1 - df.price_inr / df.list_price_inr) * 100
gap = (implied - df.discount_pct).abs()
print("Rows where |implied - stated discount| > 1pp:", (gap > 1).sum())
print("Rows where sale price > list price          :", (df.price_inr > df.list_price_inr).sum())
# Recompute from prices: it's more precise than the rounded string
df["discount_pct"] = implied.round(2)

Rows where |implied - stated discount| > 1pp: 0
Rows where sale price > list price          : 0


### 2.2 Fix the bad rating and missing review counts

In [7]:
print(df.loc[df.rating.isna() | df.rating_count.isna(), ["product_id", "rating", "rating_count", "category"]])

# Impute the rating with the median of its leaf category (falls back to the global median)
leaf = df["category"].str.split("|").str[-1]
df["rating"] = df["rating"].fillna(df.groupby(leaf)["rating"].transform("median")).fillna(df.rating.median())

# Missing rating_count: both rows are new 2022 ASINs (B0B…). Use the 10th percentile of their main category
# rather than the median, because new listings have few reviews.
main = df["category"].str.split("|").str[0]
p10 = df.groupby(main)["rating_count"].transform(lambda s: s.quantile(0.10))
df["rating_count"] = df["rating_count"].fillna(p10).round()
df[["rating", "rating_count"]].isna().sum()

      product_id  rating  rating_count                                           category
282   B0B94JPY2N     3.0           NaN  Computers&Accessories|Accessories&Peripherals|...
324   B0BQRJ3C47     5.0           NaN  Computers&Accessories|Accessories&Peripherals|...
1279  B08L12N5H1     NaN         992.0  Home&Kitchen|Kitchen&HomeAppliances|Vacuum,Cle...


rating          0
rating_count    0
dtype: int64

## 3. Merge duplicate ASINs
Duplicate rows are the same product with near-identical numbers (e.g. `rating_count` 107,687 vs 107,686).
We keep one row per ASIN: numbers come from the snapshot with the **most reviews** (the latest), and we keep the union of review text.

In [8]:
dups = df[df.product_id.duplicated(keep=False)]
print("Products with >1 row:", dups.product_id.nunique())
print("…of which price differs:", (dups.groupby("product_id").price_inr.nunique() > 1).sum())

def join_unique(s):
    parts = []
    for v in s.dropna():
        parts.extend(p for p in str(v).split(",") if p and p not in parts)
    return ",".join(parts)

df = df.sort_values("rating_count", ascending=False)
text_cols = ["review_title", "review_content", "user_id", "review_id"]
merged_text = df.groupby("product_id")[text_cols].agg(join_unique)
df = df.drop_duplicates("product_id", keep="first").set_index("product_id")
df[text_cols] = merged_text
df = df.reset_index()
print("Rows after merge:", len(df))

Products with >1 row: 92
…of which price differs: 3
Rows after merge: 1351


## 4. Category hierarchy → U.S. taxonomy
`category` is a pipe-separated path, e.g. `Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables`.
We split it into levels, then map it to the U.S. Amazon top-level category, which sets the **referral fee** and other economics later.
Some Indian sub-categories sit under different parents in the U.S. (e.g. Mobile accessories → *Cell Phones & Accessories*).

In [9]:
levels = df["category"].str.split("|", expand=True)
df["cat_l1"] = levels[0]
df["cat_l2"] = levels[1]
df["cat_l3"] = levels[2]
df["cat_leaf"] = df["category"].str.split("|").str[-1]
df["cat_depth"] = df["category"].str.count(r"\|") + 1

def us_category(row) -> str:
    l1, l2 = row.cat_l1, row.cat_l2 or ""
    if l2 == "Mobiles&Accessories":          return "Cell Phones & Accessories"
    if l2 == "WearableTechnology":           return "Cell Phones & Accessories"
    if l2 == "Headphones,Earbuds&Accessories" or l2 == "HomeAudio": return "Electronics - Audio"
    if l2 == "HomeTheater,TV&Video":         return "Electronics - TV & Video"
    if l2 == "Cameras&Photography":          return "Camera & Photo"
    if l1 == "Electronics":                  return "Electronics - Accessories"
    if l1 == "Computers&Accessories":
        return "Computers & Accessories" if l2 in ("Components", "Monitors", "Tablets", "ExternalDevices&DataStorage", "NetworkingDevices", "Laptops") else "Computer Accessories & Peripherals"
    if l1 == "Home&Kitchen":
        return "Kitchen & Home Appliances" if l2 in ("Kitchen&HomeAppliances", "Heating,Cooling&AirQuality") else "Home & Kitchen"
    if l1 == "OfficeProducts":               return "Office Products"
    if l1 == "MusicalInstruments":           return "Musical Instruments"
    if l1 == "HomeImprovement":              return "Tools & Home Improvement"
    if l1 == "Toys&Games":                   return "Toys & Games"
    if l1 == "Car&Motorbike":                return "Automotive"
    if l1 == "Health&PersonalCare":          return "Health & Household"
    return "Other"

df["us_category"] = df.apply(us_category, axis=1)
df["us_category"].value_counts()

us_category
Kitchen & Home Appliances             424
Computer Accessories & Peripherals    318
Cell Phones & Accessories             213
Electronics - TV & Video              156
Electronics - Audio                    79
Computers & Accessories                57
Office Products                        31
Electronics - Accessories              26
Home & Kitchen                         24
Camera & Photo                         16
Musical Instruments                     2
Tools & Home Improvement                2
Toys & Games                            1
Health & Household                      1
Automotive                              1
Name: count, dtype: int64

## 5. Localise prices: INR → U.S. market USD
Three steps:
1. **FX:** 1 USD ≈ ₹83.0 (the 2023–24 average RBI reference rate).
2. **U.S. market multiplier:** the same product typically lists 1.6–2.6× higher in the U.S. once you allow for purchasing-power differences,
   import duties and higher U.S. fulfilment and labour costs. Cheap accessories get the largest uplift and big-ticket electronics the smallest,
   because global brands price TVs and phones close to parity. We use a log-scaled multiplier with ±8% product-level noise.
3. **Price-point snapping:** U.S. retail clusters at `x.99` prices with a $5.99 floor (below that, Amazon "add-on" economics don't work for FBA sellers).

These are approximations, but every step is explicit and has a business reason.

In [10]:
FX_INR_PER_USD = 83.0

def us_multiplier(price_usd_fx: pd.Series) -> pd.Series:
    # ~2.6x at $2, ~2.0x at $20, ~1.6x at $200, floor 1.25x
    m = 2.6 - 0.45 * np.log10(price_usd_fx.clip(lower=2) / 2)
    return m.clip(lower=1.25)

def snap_99(x: pd.Series, floor: float = 5.99) -> pd.Series:
    return (np.floor(x.clip(lower=floor)) + 0.99).where(x >= floor, floor)

list_fx = df.list_price_inr / FX_INR_PER_USD
noise = RNG.normal(1.0, 0.08, len(df)).clip(0.85, 1.15)
mult = us_multiplier(list_fx) * noise

df["list_price_usd"] = snap_99(list_fx * mult)
# Keep the original discount depth, re-applied to the U.S. list price
df["price_usd"] = snap_99(df.list_price_usd * (1 - df.discount_pct / 100))
df["price_usd"] = np.minimum(df.price_usd, df.list_price_usd)
df["discount_pct"] = ((1 - df.price_usd / df.list_price_usd) * 100).round(2)
df["discount_usd"] = (df.list_price_usd - df.price_usd).round(2)

df[["price_inr", "list_price_inr", "price_usd", "list_price_usd", "discount_pct"]].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
price_inr,1351.0,3304.73,7173.99,39.00,349.00,899.00,2174.00,77990.00
list_price_inr,1351.0,5690.51,11218.92,39.00,899.00,1790.00,4575.00,139900.00
price_usd,1351.0,68.67,130.59,5.99,9.99,22.99,52.99,1251.99
list_price_usd,1351.0,118.54,202.22,5.99,23.99,45.99,106.99,2244.99
discount_pct,1351.0,44.46,21.45,0.00,28.44,46.74,60.03,94.01


### 5.1 Discount sanity check for the U.S.
Indian listings often show inflated "MRP" strikethroughs (discounts of 70–90%). On U.S. Amazon, a reference price must be a real recent price,
so typical discounts are 10–40% and rarely above 60%. We cap the **effective everyday discount** at ~60% by raising the list price,
and keep the original depth as a feature (`orig_discount_pct`), because the "deep-discount" signal is still useful.

In [11]:
df["orig_discount_pct"] = df["discount_pct"]
cap = 60.0
mask = df.discount_pct > cap
print(f"Listings with discount > {cap}%: {mask.sum()} ({mask.mean():.1%}). Their list price is lowered to a {cap}% reference.")
df.loc[mask, "list_price_usd"] = snap_99(df.loc[mask, "price_usd"] / (1 - cap / 100))
df["discount_pct"] = ((1 - df.price_usd / df.list_price_usd) * 100).clip(lower=0).round(2)
df["discount_usd"] = (df.list_price_usd - df.price_usd).round(2)
df[["discount_pct", "orig_discount_pct"]].describe().T.round(1)

Listings with discount > 60.0%: 346 (25.6%). Their list price is lowered to a 60.0% reference.


,count,mean,std,min,25%,50%,75%,max
discount_pct,1351.0,41.9,18.2,0.0,28.4,46.7,60.0,61.2
orig_discount_pct,1351.0,44.5,21.5,0.0,28.4,46.7,60.0,94.0


## 6. Text-derived features

In [12]:
BRAND_FIX = {"boAt": "boAt", "MI": "Xiaomi", "Redmi": "Xiaomi", "Mi": "Xiaomi"}
df["brand"] = df["product_name"].str.split().str[0].str.strip(",()").replace(BRAND_FIX)
df["name_len"] = df["product_name"].str.len()
df["about_len"] = df["about_product"].fillna("").str.len()
df["n_bullets"] = df["about_product"].fillna("").str.count(r"\|") + 1
df["n_reviews_sampled"] = df["review_id"].fillna("").str.count(",") + 1

# Simple lexicon sentiment of the sampled review titles (no NLP dependency needed)
POS = r"\b(good|great|excellent|nice|awesome|best|value|perfect|love|satisf|worth|recommend|amazing|fine|quality)"
NEG = r"\b(bad|poor|worst|waste|broken|defect|stopp|not working|cheap|return|disappoint|fake|issue|problem)"
t = df["review_title"].fillna("").str.lower()
pos, neg = t.str.count(POS), t.str.count(NEG)
df["review_sentiment"] = ((pos - neg) / (pos + neg).replace(0, np.nan)).fillna(0).round(3)

# Brand strength: number of listings the brand has in this catalogue (proxy for brand presence)
df["brand_listing_count"] = df.groupby("brand")["product_id"].transform("count")
df["is_bundle"] = df["product_name"].str.contains(r"pack of|set of|combo|\b\d+\s?(?:pcs|pieces)\b", case=False).astype(int)
df[["brand", "name_len", "n_bullets", "review_sentiment", "brand_listing_count", "is_bundle"]].head()

,brand,name_len,n_bullets,review_sentiment,brand_listing_count,is_bundle
0,AmazonBasics,73,6,1.0,27,0
1,Amazon,58,6,1.0,23,0
2,Amazon,83,5,1.0,23,0
3,boAt,63,7,0.5,52,0
4,boAt,62,7,0.5,52,0


## 7. Drop columns we don't need
| Column | Reason |
|---|---|
| `user_id`, `user_name`, `review_id` | PII / identifiers. No predictive value. |
| `discounted_price`, `actual_price`, `discount_percentage` | Raw strings, replaced by parsed USD columns |
| `price_inr`, `list_price_inr` | INR. We only keep USD for the U.S. scope (INR goes to an audit file) |
| `review_content` | Long free text. Summarised into `review_sentiment`; the full text stays in the audit file |
| `category` | Replaced by `cat_l1…cat_leaf`, `us_category` |

`img_link`, `product_link` and `about_product` go to a separate **catalogue file** for the web front-end, not the modelling table.

In [13]:
catalogue = df[["product_id", "product_name", "brand", "us_category", "about_product", "img_link", "product_link"]].copy()
catalogue["product_link"] = catalogue["product_link"].str.replace("amazon.in", "amazon.com", regex=False)
catalogue.to_csv(OUT / "product_catalogue.csv", index=False)

audit = df[["product_id", "price_inr", "list_price_inr", "orig_discount_pct", "review_title", "review_content"]]
audit.to_csv(OUT / "audit_inr_and_reviews.csv", index=False)

KEEP = ["product_id", "product_name", "brand", "us_category", "cat_l1", "cat_l2", "cat_l3", "cat_leaf", "cat_depth",
        "price_usd", "list_price_usd", "discount_pct", "discount_usd", "orig_discount_pct",
        "rating", "rating_count", "review_sentiment", "n_reviews_sampled",
        "name_len", "about_len", "n_bullets", "brand_listing_count", "is_bundle"]
clean = df[KEEP].copy()
clean["rating_count"] = clean["rating_count"].astype(int)
clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1351 entries, 0 to 1350
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_id           1351 non-null   str    
 1   product_name         1351 non-null   str    
 2   brand                1351 non-null   object 
 3   us_category          1351 non-null   str    
 4   cat_l1               1351 non-null   str    
 5   cat_l2               1351 non-null   str    
 6   cat_l3               1343 non-null   str    
 7   cat_leaf             1351 non-null   object 
 8   cat_depth            1351 non-null   int64  
 9   price_usd            1351 non-null   float64
 10  list_price_usd       1351 non-null   float64
 11  discount_pct         1351 non-null   float64
 12  discount_usd         1351 non-null   float64
 13  orig_discount_pct    1351 non-null   float64
 14  rating               1351 non-null   float64
 15  rating_count         1351 non-null   int64  
 16 

## 8. Final checks & save

In [14]:
assert clean.product_id.is_unique
assert clean.isna().sum().sum() == 0 or clean.isna().sum()[lambda s: s > 0].index.isin(["cat_l3"]).all()
assert (clean.price_usd <= clean.list_price_usd).all()
assert clean.rating.between(1, 5).all()
clean["cat_l3"] = clean["cat_l3"].fillna(clean["cat_l2"])
clean.to_csv(OUT / "amazon_us_clean.csv", index=False)
print("Saved", OUT / "amazon_us_clean.csv", clean.shape)
clean.sample(5, random_state=7)

Saved data/interim/amazon_us_clean.csv (1351, 23)


,product_id,product_name,brand,us_category,cat_l1,cat_l2,cat_l3,cat_leaf,cat_depth,price_usd,list_price_usd,discount_pct,discount_usd,orig_discount_pct,rating,rating_count,review_sentiment,n_reviews_sampled,name_len,about_len,n_bullets,brand_listing_count,is_bundle
328,B07N8RQ6W7,Portronics MODESK POR-122 Universal Mobile Tab...,Portronics,Cell Phones & Accessories,Electronics,Mobiles&Accessories,MobileAccessories,Stands,4,5.99,14.99,60.04,9.0,66.70,4.1,16685,1.0,8,66,597,5,21,0
262,B07WGMMQGP,"iQOO vivo Z6 5G (Chromatic Blue, 6GB RAM, 128G...",iQOO,Cell Phones & Accessories,Electronics,Mobiles&Accessories,Smartphones&BasicMobiles,Smartphones,4,335.99,426.99,21.31,91.0,21.31,4.0,21350,1.0,8,126,1107,5,14,0
981,B09L835C3V,"Smashtronics® - Case for Firetv Remote, Fire S...",Smashtronics®,Electronics - TV & Video,Electronics,"HomeTheater,TV&Video",Accessories,RemoteControls,4,5.99,10.99,45.50,5.0,45.50,4.2,1335,1.0,8,183,1025,4,2,0
755,B0B1F6GQPS,"Boult Audio FXCharge with ENC, 32H Playtime, 5...",Boult,Electronics - Audio,Electronics,"Headphones,Earbuds&Accessories",Headphones,In-Ear,4,25.99,64.99,60.01,39.0,77.59,3.8,3390,1.0,8,176,579,5,11,0
13,B09X7DY7Q4,SanDisk Extreme SD UHS I 64GB Card for 4K Vide...,SanDisk,Electronics - Accessories,Electronics,Accessories,MemoryCards,MicroSD,4,26.99,50.99,47.07,24.0,47.07,4.5,205052,1.0,8,107,281,4,16,0


### Summary
- **1,465 → 1,351** unique products, 0 missing values, all numeric fields typed.
- Prices localised to **U.S. market USD** (FX + market multiplier + `.99` price points; discount capped at a U.S.-realistic 60%).
- Category mapped to the **U.S. Amazon taxonomy**, used next for referral fees and cost structures.
- Next: `02_eda.ipynb`.